# 02d — Làm sạch nội dung bài và build lại 1.000 URL

Chạy trên **CPU** sau khi branch `feature/laodong-extractor` đã được push. Notebook đọc raw `stage-a-laodong-recovered-v1` hiện có, dùng quy tắc cắt nội dung phụ mới và ghi build riêng `stage-a-data-v5-clean-text`. Không sửa raw hay build v4. Chạy từ đầu trong runtime Colab mới.


In [ ]:
import importlib.util, json, os, subprocess, sys
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
DATA_ROOT = Path('/content/drive/MyDrive/VietMedBridge/data')
WORK_DIR = Path('/content/vmb_work')
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ['HF_HOME'] = '/content/hf_cache'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
REPO_URL = 'https://github.com/Platypus27-coder/VietMedBridge.git'
CODE_REF = 'feature/laodong-extractor'
CHECKOUT = Path('/content/VietMedBridge')
if 'vietmedbridge' in sys.modules:
    raise RuntimeError('Đã import package trong runtime này. Restart session rồi chạy lại 02d.')
if not (CHECKOUT / '.git').exists():
    subprocess.run(['git', 'clone', '--no-checkout', REPO_URL, str(CHECKOUT)], check=True)
if subprocess.check_output(['git', '-C', str(CHECKOUT), 'remote', 'get-url', 'origin'], text=True).strip() != REPO_URL:
    raise RuntimeError('Checkout không thuộc repo VietMedBridge')
subprocess.run(['git', '-C', str(CHECKOUT), 'fetch', '--depth', '1', 'origin', CODE_REF], check=True)
subprocess.run(['git', '-C', str(CHECKOUT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
CODE_COMMIT = subprocess.check_output(['git', '-C', str(CHECKOUT), 'rev-parse', 'HEAD'], text=True).strip()
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[notebook]'], cwd=CHECKOUT, check=True)
sys.path.insert(0, str(CHECKOUT / 'src'))
importlib.invalidate_caches()
print('Code commit:', CODE_COMMIT)


## 1. Xác minh raw 1.000 URL đã lưu

In [ ]:
from vietmedbridge.artifacts import read_json
from vietmedbridge.crawl import read_completed_crawl
from vietmedbridge.dataset import load_snapshot, parquet_path

SNAPSHOT = load_snapshot(DATA_ROOT)
PILOT = read_json(DATA_ROOT / 'reports/inventory/stage_a.json')
CRAWL_RUN = 'stage-a-laodong-recovered-v1'
BUILD_RUN = 'stage-a-data-v5-clean-text'
OLD_BUILD_RUN = 'stage-a-data-v4-quality'
RAW = read_completed_crawl(
    DATA_ROOT / 'crawl' / CRAWL_RUN,
    links_path=DATA_ROOT / PILOT['files']['stage_a_links']['path'],
    origin_corpus_sha256=SNAPSHOT['files']['links_corpus.parquet']['sha256'],
)
print('Code commit:', CODE_COMMIT)
print('Raw đã lưu:', RAW['recorded_documents'], '/', RAW['requested_input_records'])
print('Có đủ 1.000 outcome:', RAW['range_complete'])
assert RAW['range_complete'] and RAW['recorded_documents'] == 1000


## 2. Build mới từ raw cũ

In [ ]:
from vietmedbridge.build import build_corpus
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer

TOKENIZER_LOCK = read_json(DATA_ROOT / 'tokenizer_lock.json')
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(TOKENIZER_LOCK['revision'])
PIPELINE_CONFIG = json.loads((CHECKOUT / 'configs/data_pipeline.json').read_text())
BUILD = build_corpus(
    DATA_ROOT / 'crawl' / CRAWL_RUN, DATA_ROOT / 'processed',
    TOKENIZER, TOKENIZER_SPEC, run_name=BUILD_RUN,
    config=ChunkConfig(**PIPELINE_CONFIG['chunking']), work_dir=WORK_DIR,
    official_links=parquet_path(DATA_ROOT, 'links_corpus.parquet'),
)
print('Build:', BUILD_RUN)
print(json.dumps(BUILD['counts'], ensure_ascii=False, indent=2))


## 3. So sánh build mới với v4

Số document có thể giảm nếu trang chủ hoặc mã rác bị loại. Đừng coi số document lớn hơn là bằng chứng chất lượng tốt hơn.


In [ ]:
import pyarrow.dataset as ds
import pandas as pd
from vietmedbridge.health import health_report

def document_frame(run_name):
    paths = sorted((DATA_ROOT / 'processed' / run_name).rglob('*.documents.parquet'))
    if not paths:
        raise FileNotFoundError(f'Không tìm thấy documents của {run_name}')
    return ds.dataset([str(path) for path in paths], format='parquet').to_table(
        columns=['doc_id', 'url', 'title', 'language', 'language_method',
                 'quality_tier', 'quality_flags', 'source_text_sha256', 'source_text']
    ).to_pandas()

OLD = document_frame(OLD_BUILD_RUN)
NEW = document_frame(BUILD_RUN)
old_ids, new_ids = set(OLD.doc_id), set(NEW.doc_id)
print('V4 documents:', len(OLD), '| V5 documents:', len(NEW))
print('Không còn trong V5:', len(old_ids - new_ids), sorted(old_ids - new_ids)[:30])
print('Mới trong V5:', len(new_ids - old_ids), sorted(new_ids - old_ids)[:30])
print('Ngôn ngữ V5:')
display(NEW.language.value_counts().rename_axis('language').reset_index(name='documents'))
print('Quality tier V5:')
display(NEW.quality_tier.value_counts().rename_axis('quality_tier').reset_index(name='documents'))
HEALTH = health_report(
    DATA_ROOT / 'processed' / BUILD_RUN,
    official_links=parquet_path(DATA_ROOT, 'links_corpus.parquet'),
    crawl_dir=DATA_ROOT / 'crawl' / CRAWL_RUN,
    work_dir=WORK_DIR, audit_size=PIPELINE_CONFIG['audit_size'],
)
print('Failure reasons:')
display(pd.DataFrame(HEALTH['failure_reasons']))
print('Health report:', DATA_ROOT / 'processed' / BUILD_RUN / 'reports')


## 4. Kiểm tra dấu hiệu nội dung phụ trong V4 và V5

Các số sau là dấu hiệu tự động, không thay cho việc xem lại một số bài theo domain.


In [ ]:
import re
from collections import Counter

MARKERS = {
    'youlai_doctor': r'医生推荐',
    'familydoctor_related': r'相关文章推荐|相关推荐|相关问答',
    'medlatec_service': r'Bình luận \(\)|Lựa chọn dịch vụ',
    'phutho_template': r'\{(?:title|publish|head|name|body)\}',
    'pharmacity_related': r'Các bài viết liên quan',
    'nghean_related': r'\| TIN LIÊN QUAN \|',
    'replacement_char': '�',
}

def marker_counts(frame):
    return {name: int(frame.source_text.str.contains(pattern, flags=re.I, regex=True, na=False).sum())
            for name, pattern in MARKERS.items()}

print('V4 document markers:', marker_counts(OLD))
print('V5 document markers:', marker_counts(NEW))
child_paths = sorted((DATA_ROOT / 'processed' / BUILD_RUN).rglob('*.children.parquet'))
children = ds.dataset([str(path) for path in child_paths], format='parquet').to_table(
    columns=['doc_id', 'text']).to_pandas()
print('V5 chunks:', len(children))
print('V5 chunk markers:', {name: int(children.text.str.contains(pattern, flags=re.I, regex=True, na=False).sum())
                           for name, pattern in MARKERS.items()})
print('V5 flags:', Counter(flag for flags in NEW.quality_flags for flag in flags).most_common(15))


## Đọc kết quả

Gửi bảng `V4 documents / V5 documents`, `Ngôn ngữ V5`, `Quality tier V5` và `Failure reasons` để đánh giá. Nếu cần xem từng bài bị loại, dùng `OLD[OLD.doc_id.isin(old_ids - new_ids)]` trong ô mới. Sau đó chạy notebook 03 với `CRAWL_RUN = 'stage-a-laodong-recovered-v1'`, `BUILD_RUN = 'stage-a-data-v5-clean-text'`; nếu notebook 03 có ô audit tự thêm từng trỏ cố định vào build v2/v3, sửa đường dẫn trong ô đó sang v5. Chưa chạy 10k chỉ dựa vào số lượng document.

Ô kiểm tra ngay trên đếm các dấu hiệu nội dung phụ trong toàn bộ document và chunk của V4/V5. Chỉ nâng lên 10.000 URL sau khi xem cả số bài giữ được và các dấu hiệu còn sót.
